# Foreground IOU Evaluation Pipeline (Single Class)
> Compare predicted masks from fine-tuned SAM3 model with ground truth masks for single category 

In [ ]:
#| default_exp evaluation.single_eval_pipe

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:

# Path to  test images
TEST_IMAGES_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/source_images')

# Path to ground truth masks
GT_MASKS_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/source_masks')

# Model paths
BPE_PATH = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
SAM3_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
FINETUNED_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3_data/experiments/decoder_data/I1000E200/checkpoints/checkpoint_200.pt')

# Inference settings
TEXT_PROMPT = "pin"
CONFIDENCE_THRESHOLD = 0.5

# Output directory
OUTPUT_DIR = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/1000images/evaluation_output')

In [ ]:
#| export
def load_model(bpe_path: str, sam3_checkpoint: Path, finetuned_checkpoint: Path, 
               device: torch.device = None) -> torch.nn.Module:
    """Load SAM3 model with fine-tuned weights."""
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = build_sam3_image_model(bpe_path=bpe_path, checkpoint_path=sam3_checkpoint)
    finetuned_weights = torch.load(finetuned_checkpoint, map_location=device)
    model.load_state_dict(finetuned_weights['model'])
    model.eval()
    
    print("Model loaded")
    return model

In [ ]:
# Load the model
model = load_model(
    bpe_path=BPE_PATH,
    sam3_checkpoint=SAM3_CHECKPOINT,
    finetuned_checkpoint=FINETUNED_CHECKPOINT
)

>Evaluation metrics

In [ ]:
#| export
def compute_foreground_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> dict:
    """
    Compute Foreground IoU between predicted and ground truth masks.
    
    Foreground IoU = TP / (TP + FP + FN)
    
    Args:
        pred_mask: Predicted binary mask (H, W) - 1 for foreground, 0 for background
        gt_mask: Ground truth binary mask (H, W) - 1 for foreground, 0 for background
    
    Returns:
        Dictionary with foreground_iou, TP, FP, FN, TN pixel counts
    """
    pred_mask = pred_mask.astype(bool)
    gt_mask = gt_mask.astype(bool)
    
    # True Positive: Both predict and GT are foreground
    TP = np.logical_and(pred_mask, gt_mask).sum()
    
    # False Positive: Predicted foreground but GT is background
    FP = np.logical_and(pred_mask, ~gt_mask).sum()
    
    # False Negative: Predicted background but GT is foreground
    FN = np.logical_and(~pred_mask, gt_mask).sum()
    
    # True Negative: Both predict and GT are background
    TN = np.logical_and(~pred_mask, ~gt_mask).sum()
    
    # Foreground IoU
    denominator = TP + FP + FN
    if denominator == 0:
        fg_iou = 1.0  # No foreground in either - perfect match
    else:
        fg_iou = TP / denominator
    
    return {
        'foreground_iou': fg_iou,
        'TP': int(TP),
        'FP': int(FP),
        'FN': int(FN),
        'TN': int(TN),
        'precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
        'recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
    }

In [ ]:
#| export
def load_ground_truth_mask(mask_path: Path) -> np.ndarray:
    """Load ground truth mask from file."""
    mask = np.array(Image.open(mask_path).convert('L'))
    return (mask > 127).astype(np.uint8)


In [ ]:
#| export
def get_predicted_mask(model, image: Image.Image, prompt: str, 
                       confidence_threshold: float = 0.5) -> np.ndarray:
    """Run inference and get combined predicted mask."""
    processor = Sam3Processor(model, confidence_threshold=confidence_threshold)
    inference_state = processor.set_image(image)
    processor.reset_all_prompts(inference_state)
    inference_state = processor.set_text_prompt(state=inference_state, prompt=prompt)
    
    masks = inference_state.get('masks', [])
    
    if len(masks) == 0:
        return np.zeros((image.height, image.width), dtype=np.uint8)
    
    combined_mask = np.zeros((image.height, image.width), dtype=bool)
    for mask in masks:
        if isinstance(mask, torch.Tensor):
            mask = mask.cpu().numpy()
        if mask.ndim == 3:
            mask = mask.squeeze()
        combined_mask = np.logical_or(combined_mask, mask > 0.5)
    
    return combined_mask.astype(np.uint8)


In [ ]:
#| export
def find_matching_mask(image_path: Path, masks_dir: Path) -> Path:
    """Find the corresponding ground truth mask for an image."""
    stem = image_path.stem
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']:
        mask_path = masks_dir / f"{stem}{ext}"
        if mask_path.exists():
            return mask_path
    raise FileNotFoundError(f"No mask found for {image_path.name} in {masks_dir}")

>Main Evaluation

In [ ]:
#| export
def evaluate_foreground_iou(model, images_dir: Path, masks_dir: Path, 
                             prompt: str, confidence_threshold: float = 0.5,
                             start_idx: int = 0, end_idx: int = None,
                             max_samples: int = None) -> dict:
    """
    Evaluate model using Foreground IoU metric.
    
    Args:
        model: SAM3 model
        images_dir: Directory containing test images
        masks_dir: Directory containing ground truth masks
        prompt: Text prompt for detection
        confidence_threshold: Confidence threshold for model
        start_idx: Start index for image selection
        end_idx: End index for image selection
        max_samples: Maximum number of samples to evaluate
    
    Returns:
        Dictionary with evaluation results
    """
    all_images = sorted(list(images_dir.glob('*.png')))
    
    if end_idx is None:
        end_idx = len(all_images)
    
    image_files = all_images[start_idx:end_idx]
    
    if max_samples:
        image_files = image_files[:max_samples]
    
    print(f"Evaluating {len(image_files)} images (index {start_idx} to {end_idx})")
    
    results = {
        'foreground_iou': [],
        'precision': [],
        'recall': [],
        'TP_pixels': [],
        'FP_pixels': [],
        'FN_pixels': [],
        'TN_pixels': [],
        'gt_fg_pixels': [],
        'pred_fg_pixels': [],
        'file_names': [],
        'errors': []
    }
    
    for img_path in tqdm(image_files, desc="Evaluating Foreground IoU"):
        try:
            # Load image
            image = Image.open(img_path).convert('RGB')
            
            # Find and load ground truth mask
            mask_path = find_matching_mask(img_path, masks_dir)
            gt_mask = load_ground_truth_mask(mask_path)
            
            # Get prediction
            with torch.no_grad():
                pred_mask = get_predicted_mask(model, image, prompt, confidence_threshold)
            
            # Resize prediction if needed
            if pred_mask.shape != gt_mask.shape:
                pred_mask = np.array(
                    Image.fromarray(pred_mask).resize(
                        (gt_mask.shape[1], gt_mask.shape[0]), 
                        Image.NEAREST
                    )
                )
            
            # Compute Foreground IoU
            fg_metrics = compute_foreground_iou(pred_mask, gt_mask)
            
            results['foreground_iou'].append(fg_metrics['foreground_iou'])
            results['precision'].append(fg_metrics['precision'])
            results['recall'].append(fg_metrics['recall'])
            results['TP_pixels'].append(fg_metrics['TP'])
            results['FP_pixels'].append(fg_metrics['FP'])
            results['FN_pixels'].append(fg_metrics['FN'])
            results['TN_pixels'].append(fg_metrics['TN'])
            results['gt_fg_pixels'].append(int(gt_mask.sum()))
            results['pred_fg_pixels'].append(int(pred_mask.sum()))
            results['file_names'].append(img_path.name)
            
        except Exception as e:
            results['errors'].append((img_path.name, str(e)))
            print(f"Error processing {img_path.name}: {e}")
    
    return results

In [ ]:

results = evaluate_foreground_iou(
    model=model,
    images_dir=TEST_IMAGES_PATH,
    masks_dir=GT_MASKS_PATH,
    prompt=TEXT_PROMPT,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    start_idx=3000,      
    end_idx=3020,        
    max_samples=None
)

In [ ]:
#| export
def calc_foreground_iou_summary(results: dict) -> dict:
    """Print and return summary statistics for Foreground IoU evaluation."""
    n_samples = len(results['foreground_iou'])
    n_errors = len(results['errors'])
    
    print("FOREGROUND IoU EVALUATION SUMMARY")
    print(f"Total images evaluated: {n_samples}")
    print(f"Errors: {n_errors}")
    
    summary_dict = {
        'total_images': n_samples,
        'errors': n_errors,
    }
    
    if n_samples > 0:
        # Aggregate pixel counts
        total_TP = sum(results['TP_pixels'])
        total_FP = sum(results['FP_pixels'])
        total_FN = sum(results['FN_pixels'])
        total_TN = sum(results['TN_pixels'])
        
        # Overall Foreground IoU (from aggregated pixels)
        overall_fg_iou = total_TP / (total_TP + total_FP + total_FN) if (total_TP + total_FP + total_FN) > 0 else 0
        
        # Mean metrics
        mean_fg_iou = np.mean(results['foreground_iou'])
        mean_precision = np.mean(results['precision'])
        mean_recall = np.mean(results['recall'])
        
        # Pixel-level precision/recall
        pixel_precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0
        pixel_recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0
        pixel_f1 = 2 * pixel_precision * pixel_recall / (pixel_precision + pixel_recall) if (pixel_precision + pixel_recall) > 0 else 0
        
        # Find best and worst
        sorted_indices = np.argsort(results['foreground_iou'])
        worst_idx = sorted_indices[0]
        best_idx = sorted_indices[-1]
        
        print("FOREGROUND IoU METRICS")
        print(f"\nOVERALL (Aggregated Pixels):")
        print(f"Foreground IoU:{overall_fg_iou*100:6.2f}%")
        print(f"Precision:{pixel_precision*100:6.2f}% | Recall:{pixel_recall*100:6.2f}% | F1 Score:{pixel_f1*100:6.2f}% ")
        #print(f"Recall:{pixel_recall*100:6.2f}%")
        #print(f"F1 Score:{pixel_f1*100:6.2f}%")
        
        print(f"\nMEAN (Per-Image Average):")
        print(f"Mean FG IoU:{mean_fg_iou*100:6.2f}% | Mean Precision:{mean_precision*100:6.2f}% |Mean Recall:{mean_recall*100:6.2f}%")
        #print(f"Mean Precision:{mean_precision*100:6.2f}%")
        #print(f"Mean Recall:{mean_recall*100:6.2f}%")
        
        print(f"\nPIXEL COUNT BREAKDOWN:")
        print(f"TP: {total_TP:,} pixels")
        print(f"FP: {total_FP:,} pixels")
        print(f"FN: {total_FN:,} pixels")
        print(f"TN: {total_TN:,} pixels")
        
        print(f"\nBEST & WORST SAMPLES:")
        print(f"Best:  {results['file_names'][best_idx]} (FG IoU: {results['foreground_iou'][best_idx]:.4f})")
        print(f"Worst: {results['file_names'][worst_idx]} (FG IoU: {results['foreground_iou'][worst_idx]:.4f})")
        
        summary_dict.update({
            'overall_foreground_iou': round(overall_fg_iou * 100, 2),
            'mean_foreground_iou': round(mean_fg_iou * 100, 2),
            'pixel_precision': round(pixel_precision * 100, 2),
            'pixel_recall': round(pixel_recall * 100, 2),
            'pixel_f1': round(pixel_f1 * 100, 2),
            'total_TP_pixels': total_TP,
            'total_FP_pixels': total_FP,
            'total_FN_pixels': total_FN,
            'total_TN_pixels': total_TN,
            'best_sample': {
                'file_name': results['file_names'][best_idx],
                'foreground_iou': round(results['foreground_iou'][best_idx], 4),
            },
            'worst_sample': {
                'file_name': results['file_names'][worst_idx],
                'foreground_iou': round(results['foreground_iou'][worst_idx], 4),
            },
        })
    
    return summary_dict

In [ ]:
# Print summary
summary = calc_foreground_iou_summary(results)

In [ ]:
#| export
def save_summary_to_json(summary: dict, results: dict, output_dir: Path, 
                          filename: str = "foreground_iou_summary.json"):
    """
    Save evaluation summary and per-image results to a JSON file.
    
    Args:
        summary: Summary dictionary from calc_foreground_iou_summary()
        results: Full results dictionary from evaluate_foreground_iou()
        output_dir: Directory to save the JSON file
        filename: Name of the output JSON file
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / filename
    
    # Add per-image details to summary
    per_image_results = []
    for i in range(len(results['file_names'])):
        per_image_results.append({
            'file_name': results['file_names'][i],
            'foreground_iou': round(results['foreground_iou'][i], 4),
            'precision': round(results['precision'][i], 4),
            'recall': round(results['recall'][i], 4),
            'TP_pixels': results['TP_pixels'][i],
            'FP_pixels': results['FP_pixels'][i],
            'FN_pixels': results['FN_pixels'][i],
            'gt_fg_pixels': results['gt_fg_pixels'][i],
            'pred_fg_pixels': results['pred_fg_pixels'][i],
        })
    
    output_data = {
        'summary': summary,
        'per_image_results': per_image_results,
        'errors': results['errors']
    }
    
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    return output_path

In [ ]:
# Save the summary
save_summary_to_json(summary, results, output_dir=OUTPUT_DIR)

In [ ]:
#| export
if __name__ == '__main__':
    # Paths
    TEST_IMAGES_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/source_images')
    GT_MASKS_PATH = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/source_masks')
    OUTPUT_DIR = Path('/home/hpc_storage/data/projects/userA/sam3_data/pins_dataset/1000images/evaluation_output')
    
    BPE_PATH = '/home/hpc_storage/data/projects/userA/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
    SAM3_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3/models--facebook--sam3/sam3.pt')
    FINETUNED_CHECKPOINT = Path('/home/hpc_storage/data/projects/userA/sam3_data/experiments/decoder_data/I1000E200/checkpoints/checkpoint_200.pt')

    TEXT_PROMPT = "pin"
    CONFIDENCE_THRESHOLD = 0.5

    # Load the model
    model = load_model(
        bpe_path=BPE_PATH,
        sam3_checkpoint=SAM3_CHECKPOINT,
        finetuned_checkpoint=FINETUNED_CHECKPOINT
    )

    #run evaluation
    results = evaluate_foreground_iou(
        model=model,
        images_dir=TEST_IMAGES_PATH,
        masks_dir=GT_MASKS_PATH,
        prompt=TEXT_PROMPT,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        start_idx=3000,     
        end_idx=3020,        
        max_samples=None
    )

    summary = calc_foreground_iou_summary(results)
    save_summary_to_json(summary, results)

In [ ]:
# VISUALIZE best-worst IMAGE WITH FOREGROUND IoU

def visualize_foreground_iou(image_path: str, model, prompt: str = TEXT_PROMPT,
                              masks_dir: Path = GT_MASKS_PATH,
                              confidence_threshold: float = 0.5):
    """
    Visualize a single image with Foreground IoU metrics.
    """
    image_path = Path(image_path)
    
    # Load image and masks
    image = Image.open(image_path).convert('RGB')
    mask_path = find_matching_mask(image_path, masks_dir)
    gt_mask = load_ground_truth_mask(mask_path)
    
    with torch.no_grad():
        pred_mask = get_predicted_mask(model, image, prompt, confidence_threshold)
    
    if pred_mask.shape != gt_mask.shape:
        pred_mask = np.array(
            Image.fromarray(pred_mask).resize(
                (gt_mask.shape[1], gt_mask.shape[0]), Image.NEAREST
            )
        )
    
    # Compute metrics
    fg_metrics = compute_foreground_iou(pred_mask, gt_mask)
    
    # Create figure
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(gt_mask, cmap='gray')
    axes[1].set_title(f'Ground Truth\n({gt_mask.sum():,} fg pixels)')
    axes[1].axis('off')
    
    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title(f'Predicted\n({pred_mask.sum():,} fg pixels)')
    axes[2].axis('off')
    
    # Overlay
    overlay = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    overlay[np.logical_and(pred_mask, gt_mask)] = [0, 255, 0]       # TP (green)
    overlay[np.logical_and(pred_mask, ~gt_mask.astype(bool))] = [255, 0, 0]   # FP (red)
    overlay[np.logical_and(~pred_mask.astype(bool), gt_mask)] = [0, 0, 255]   # FN (blue)
    
    axes[3].imshow(overlay)
    axes[3].set_title(f'FG IoU: {fg_metrics["foreground_iou"]:.3f}')
    axes[3].axis('off')
    
    plt.suptitle(f'{image_path.name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f" (Green)True Positive={fg_metrics['TP']:,} pixels | (Red)False Positive={fg_metrics['FP']:,} pixels | (Blue)False Negative={fg_metrics['FN']:,} pixels")
    print(f"\n  Foreground IoU: {fg_metrics['foreground_iou']:.4f} | Precision: {fg_metrics['precision']:.4f} | Recall: {fg_metrics['recall']:.4f}")

In [ ]:
# Visualize best and worst samples
if len(results['foreground_iou']) > 0:
    sorted_indices = np.argsort(results['foreground_iou'])
    worst_idx = sorted_indices[0]
    best_idx = sorted_indices[-1]
    
    print("WORST performing sample:")
    visualize_foreground_iou(TEST_IMAGES_PATH / results['file_names'][worst_idx], model)
    
    print("\nBEST performing sample:")
    visualize_foreground_iou(TEST_IMAGES_PATH / results['file_names'][best_idx], model)

In [ ]:
# visualize image by name
def visualize_image(image_name: str):
    """
    Visualize any image by its filename.
    
    Args:
        image_name: Name of the image file (e.g., 'image_001.png')
    """
    image_path = TEST_IMAGES_PATH / image_name
    visualize_foreground_iou(
        image_path=image_path,
        model=model,
        prompt=TEXT_PROMPT,
        masks_dir=GT_MASKS_PATH,
        confidence_threshold=CONFIDENCE_THRESHOLD
    )


# Example usage:
visualize_image("81643559_VFV4.1.5.2_2023010404103034_ID_00428049398816435592250_In_17_r_1_FRONT_Pass_image1_mean_0_2_4.png")

In [ ]:
CURRENT_NB='/home/hpc_storage/data/projects/userA/sam3_sy/nbs'
import os
os.chdir(CURRENT_NB)

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
def foo(): pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()